# AgriShield

End-to-end soil-risk pipeline: harmonise surveys, train a classifier, query live satellite + climate for a new site.

```
LUCAS + WoSIS  ->  data/agrishield_training.csv  ->  XGBoost (tuned)
New lat/lon    ->  GEE (Sentinel-2, WorldClim, static soil)  ->  risk report
```

**Hard rule this notebook follows:** every column in `FEATURE_COLUMNS` (agrishield/config.py) must be obtainable live, from lat/lon alone, with no lab test. Lab-only chemistry (OC, N, P, K, EC, CEC, bulk density) is the TARGET, never a feature.

Run cells top to bottom. Enrichment (2b) auto-skips itself if the training table already has satellite/climate columns, so re-running this notebook top-to-bottom after the first full run will NOT repeat the slow GEE pass or touch your data.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from agrishield.config import TRAINING_CSV, FEATURE_COLUMNS, EXAMPLE_SITE
from agrishield.dataset import build_training_csv
from agrishield.climate import enrich_training_csv
from agrishield.model import train, save_model, predict_proba
from agrishield.inference import live_features, live_model_inputs

pd.set_option("display.max_columns", 40)
print("root:", ROOT)
print("training table exists:", TRAINING_CSV.exists())

## 2. Base training table -- LUCAS + WoSIS only

No APIs yet, pure local files. Fast (seconds, not minutes).

This cell NEVER rebuilds if `TRAINING_CSV` already exists on disk -- it just loads it. `build_training_csv()` only runs the first time, when there's nothing there yet. This is deliberate: `build_training_csv()` only knows how to produce the raw 38-column merge, so calling it again on an already-enriched file would silently throw away every satellite/climate column you already paid GEE quota for.

`sample_year` is kept as a lookup key for step 2b -- it decides which satellite image to fetch for each row. It is never a model feature; check `FEATURE_COLUMNS` above if you want to confirm.

In [ ]:
if TRAINING_CSV.exists():
    df = pd.read_csv(TRAINING_CSV, low_memory=False)
    print("loaded existing training table (NOT rebuilt) -- rows:", len(df), " cols:", df.shape[1])
else:
    df = build_training_csv()
    print("built fresh training table -- rows:", len(df), " cols:", df.shape[1])
print(df["source"].value_counts())
display(df.head())

### 2a. Know your date coverage before spending GEE quota

Sentinel-2 (the satellite you're querying in 2b) only exists from 2015 onward. Rows with an older or missing `sample_year` will end up with satellite columns as `NaN` -- expected, not a bug. WorldClim and static soil/elevation don't depend on date, so they'll still fill in for almost every row.

In [ ]:
has_year = df["sample_year"].notna()
print("missing sample_year:", df["sample_year"].isna().sum(),
      f"({df['sample_year'].isna().mean()*100:.1f}%)")
print("year < 2015 (pre-Sentinel-2):", (df.loc[has_year, "sample_year"] < 2015).sum())
print("year >= 2015 (real spectral data possible):", (df.loc[has_year, "sample_year"] >= 2015).sum())

## 2b. Attach satellite + climate + soil columns

Three things get attached to the SAME rows as new columns:
- **Sentinel-2 bands + NDVI** -- batched per `sample_year` (one composite image per year, not one API call per row). Only fills for rows with `sample_year >= 2016` (rows before Sentinel-2 launched are skipped entirely -- not a failure, just physically impossible).
- **Static soil texture + elevation** -- from OpenLandMap/SRTM. Fills for virtually every row with coordinates, and OVERWRITES the LUCAS lab-measured `clay_pct`/`sand_pct`/`silt_pct`/`elevation_m` so training and live inference read from the identical source.
- **WorldClim** -- climatology, fills for virtually every row.

This cell auto-detects whether enrichment already ran (checks for `B2`/`tmean_c` with real values) and skips the slow GEE pass if so -- safe to re-run top-to-bottom any time.

In [ ]:
probe_cols = ["B2", "tmean_c"]
already_enriched = all(c in df.columns for c in probe_cols) and df[probe_cols].notna().any().all()

if already_enriched:
    print("training table already enriched -- skipping GEE calls")
else:
    df = enrich_training_csv(batch_size=400, max_rows=None)
    print("enrichment complete -- rows:", len(df), " cols:", df.shape[1])

## 3. Train

Loads from disk (not the in-memory `df` above) so this cell works even if you restarted the kernel after the overnight run finished. This is the production model -- tuned XGBoost, trained on all rows (satellite-era and pre-satellite alike; non-satellite rows just get median-imputed band values). Hyperparameters live in `model.py`'s `_TUNED_XGB_PARAMS`; see section 7 below if you want to re-search them.

In [ ]:
df = pd.read_csv(TRAINING_CSV, low_memory=False)
print("training on", len(df), "rows")
model, report = train(df)
print(report)
save_model(model)

cols = [c for c in FEATURE_COLUMNS if c in df.columns]
importances = pd.Series(model.named_steps["clf"].feature_importances_, index=cols).sort_values(ascending=False)
print("\nfeature importances:")
print(importances)

### 3b. Optional comparison: satellite-era rows only (`sample_year >= 2015`)

Same model, trained on just the rows that have REAL (not imputed) spectral bands, so you can see whether accuracy actually improves on cleaner data or whether the extra pre-satellite rows were pulling their weight.

In [ ]:
df_recent = df[df["sample_year"] >= 2015].copy()
print("rows with real satellite era data:", len(df_recent))
model_recent, report_recent = train(df_recent)
print("--- full dataset ---")
print(report)
print("--- satellite-era only ---")
print(report_recent)

## 4. Live inference for a new site

This is what actually runs when a farmer taps Calculate -- one coordinate, live GEE + weather calls, no training data touched. `live_model_inputs` feeds the model; `live_features` wraps that plus current weather for a human-facing report.

In [ ]:
site = EXAMPLE_SITE
report_data = live_features(site["latitude"], site["longitude"])
print(site["name"], site["latitude"], site["longitude"])
report_data

In [ ]:
inputs = live_model_inputs(site["latitude"], site["longitude"])
print(inputs)
predict_proba(model, inputs)

## 5. Diagnostics

Not part of the production path -- these cells check the training table and the model's behaviour before you trust it. Keep them; they're what catch a silently-broken enrichment run or a model that's leaning on geography instead of soil physics.

In [ ]:
df = pd.read_csv(TRAINING_CSV, low_memory=False)
print("total rows:", len(df))

sentinel_cols = ["B2","B3","B4","B5","B6","B7","B8","B11","B12","ndvi"]
static_cols   = ["clay_pct","sand_pct","silt_pct","elevation_m","slope_deg"]
climate_cols  = ["tmean_c","precip_mm","temp_seasonality","precip_seasonality"]

print("\n--- coverage (fraction non-null) ---")
print(df[sentinel_cols + static_cols + climate_cols].notna().mean().round(3).to_string())

print("\nrows with at least one real sentinel band:", df[sentinel_cols].notna().any(axis=1).sum())
print("rows with sample_year >= 2016:", (df["sample_year"] >= 2016).sum())

In [ ]:
sat_subset = df[df["B2"].notna()].copy()
print("rows with real satellite data:", len(sat_subset))

sat_cols = ["B2","B3","B4","B5","B6","B7","B8","B11","B12","ndvi"]

# WITH satellite bands
model_with, report_with = train(sat_subset)

# WITHOUT -- same rows, just drop the band columns before training
sat_subset_no_bands = sat_subset.drop(columns=[c for c in sat_cols if c in sat_subset.columns])
model_without, report_without = train(sat_subset_no_bands)

print("--- WITH satellite bands ---")
print(report_with)
print("--- WITHOUT satellite bands (same rows) ---")
print(report_without)

In [ ]:
print(df["continent"].value_counts(dropna=False))

## 6. Continent generalization check

Trains on every continent but one, tests on the one left out. This is the honest number -- in-distribution accuracy (section 3) tells you how well the model fits data like what it trained on; this tells you how it does on a region it has genuinely never seen. Expect lower recall on the acidic class here than in section 3 -- that gap is the model leaning partly on regional climate/texture signature rather than pure soil physics, and it's the main thing more balanced regional data (Africa/Asia/South America) should improve.

In [ ]:
from agrishield.model import continent_holdout_check
continent_holdout_check(df)

## 7. Hyperparameter search (optional, slow)

30 candidates x 5-fold grouped CV = 150 full fits on the whole table -- this can take a long time on a laptop. Only run it when you actually want to re-tune (e.g. after adding a lot of new regional data, since the best settings for the current class/region balance may not be best for a different one). Scoring uses `fbeta` with `beta=1.5`, which deliberately favors recall on the acidic class over precision -- missing real acidic soil matters more here than a false alarm. If that trade-off should be different for your use case, change `beta` before running.

This cell only searches and prints results -- it does NOT update `model.py`. If a search finds something better, copy the printed `best params` into `model.py`'s `_TUNED_XGB_PARAMS` by hand, then use the next cell to confirm it's actually an improvement before trusting it.

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import make_scorer, fbeta_score
from scipy.stats import randint, uniform
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
import xgboost as xgb
from agrishield.config import TARGET_COLUMN
from agrishield.model import available_features

cols = available_features(df)
work = df.dropna(subset=[TARGET_COLUMN]).copy()
X, y, groups = work[cols], work[TARGET_COLUMN], work["sample_id"]

neg, pos = (y == 0).sum(), (y == 1).sum()
base_scale = neg / pos

param_dist = {
    "clf__n_estimators": randint(150, 500),
    "clf__max_depth": randint(3, 8),
    "clf__learning_rate": uniform(0.03, 0.27),
    "clf__subsample": uniform(0.6, 0.4),
    "clf__colsample_bytree": uniform(0.6, 0.4),
    "clf__min_child_weight": randint(1, 10),
    "clf__scale_pos_weight": uniform(base_scale * 0.7, base_scale * 0.6),
}

pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("clf", xgb.XGBClassifier(random_state=0, n_jobs=-1, eval_metric="logloss")),
])

recall_scorer = make_scorer(fbeta_score, beta=1.5)  # >1 favors recall over precision
cv = GroupKFold(n_splits=5)

search = RandomizedSearchCV(
    pipe, param_distributions=param_dist, n_iter=30,
    scoring=recall_scorer, cv=cv, n_jobs=-1, random_state=0, verbose=2,
)
search.fit(X, y, groups=groups)
print("best params:", search.best_params_)
print("best CV fbeta(1.5):", search.best_score_)

### 7b. Confirm a candidate is actually better before adopting it

Compares whatever is currently deployed in `model.py` against a candidate you fill in below. Leave `CANDIDATE_PARAMS` empty to just re-check the current default's numbers.

In [ ]:
from agrishield.model import train, continent_holdout_check

CANDIDATE_PARAMS = {
    # paste the next search's "best params" here before running this cell.
    # leaving this empty just re-prints the current deployed default's numbers.
}

print("=" * 60)
print("CURRENT DEFAULT (model.py _TUNED_XGB_PARAMS)")
print("=" * 60)
_, report_current = train(df)
print(report_current)
continent_holdout_check(df)

if CANDIDATE_PARAMS:
    print("\n" + "=" * 60)
    print("CANDIDATE")
    print("=" * 60)
    _, report_candidate = train(df, params=CANDIDATE_PARAMS)
    print(report_candidate)
    continent_holdout_check(df, params=CANDIDATE_PARAMS)
else:
    print("\nno CANDIDATE_PARAMS set -- nothing to compare against. "
          "Fill it in with a real candidate before running this comparison.")

In [ ]:
from agrishield.farmer_report import _load_crop_table, suggest_crops
crops = _load_crop_table()
print(len(crops), "crops loaded")
print(crops.head())

# fake a prediction to test suggest_crops without hitting GEE
fake_prediction = {"predicted": 1, "probability": {0: 0.3, 1: 0.7}}
print(suggest_crops(fake_prediction))

In [ ]:
"""Run this to confirm crop data is actually wired up end to end."""
from agrishield.model import load_model, predict_proba
from agrishield.inference import live_features, live_model_inputs
from agrishield.farmer_report import farmer_report, suggest_crops

model = load_model()

lat, lon = -37.91, 145.13  # swap in any real coordinate to test

inputs = live_model_inputs(lat, lon)
full = live_features(lat, lon)
result = predict_proba(model, inputs)

print("=== raw prediction ===")
print(result)

print("\n=== no crop given -- suggestion mode ===")
report_no_crop = farmer_report(lat, lon, model, inputs, result, full.get("weather_now"))
print(report_no_crop)
print("suggested crops for this soil:", suggest_crops(result))

print("\n=== farmer names a specific crop ===")
report_with_crop = farmer_report(lat, lon, model, inputs, result, full.get("weather_now"), crop="potato")
print(report_with_crop)

In [ ]:
print(df["oc_gkg"].describe())
print("non-null:", df["oc_gkg"].notna().sum(), "/", len(df))

In [ ]:
# NEW CELL 1 -- train the regression model, separate from the classifier above
from agrishield.model import train_regression

oc_pipe, oc_metrics = train_regression(df)
print("organic carbon regression metrics:")
print(oc_metrics)

In [ ]:
# NEW CELL 2 -- the same discipline we used for the classifier: don't trust
# the number above until you've checked it holds up region by region
from agrishield.model import continent_holdout_regression

continent_holdout_regression(df)

In [ ]:
# NEW CELL 3 -- ONLY save if the metrics above look reasonable. Note the
# DIFFERENT filename -- this must not overwrite soil_risk_model.joblib
from agrishield.model import save_model
from agrishield.config import MODELS_DIR

save_model(oc_pipe, path=MODELS_DIR / "oc_model.joblib")

In [ ]:
# NEW CELL 4 -- end-to-end sanity check, same site you've used before
from agrishield.inference import live_model_inputs
from agrishield.model import predict_oc
from agrishield.farmer_report import farmer_report
from agrishield.config import EXAMPLE_SITE

inputs = live_model_inputs(EXAMPLE_SITE["latitude"], EXAMPLE_SITE["longitude"])
oc_value = predict_oc(oc_pipe, inputs)
print("predicted oc_gkg:", oc_value)

report = farmer_report(
    lat=EXAMPLE_SITE["latitude"], lon=EXAMPLE_SITE["longitude"],
    model=model, live_inputs=inputs, prediction=predict_proba(model, inputs),
    predicted_oc=oc_value,
)
print(report["oc_note"], "|", report["predicted_oc_gkg"])